# 07 · Python & Web Search: HDB Prices 🏠🐍🔍

Two **built-in tools** — OpenAI runs them for you, no Python function needed:

| Tool | What it does |
|---|---|
| **python tool** (`code_interpreter`) | writes and runs Python on `hdb_resale_prices.csv` — every HDB resale flat sold in the last 12 months (from data.gov.sg) |
| **web search** | looks up anything that changes, like news |

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-6-luna"
print("Ready!")

In [ ]:
bot = "You are SG Data Bot, a helpful assistant for life in Singapore. Keep replies short."

## 1. A quick look at the data (pandas)

In [ ]:
import pandas as pd

hdb = pd.read_csv("hdb_resale_prices.csv")
print(len(hdb), "flats sold")
hdb.head()

That's a lot of rows to paste into a chat! Instead, we **upload** the file to OpenAI and let the AI analyse it with Python.

## 2. Upload the CSV

In [ ]:
hdb_file = client.files.create(file=open("hdb_resale_prices.csv", "rb"), purpose="user_data")

print(hdb_file.id)

## 3. The python tool 🐍
We give the python tool a **container** (a little computer at OpenAI) with our file inside.

In [ ]:
python_tool = {"type": "code_interpreter", "container": {"type": "auto", "file_ids": [hdb_file.id]}}

response = client.responses.create(
    model=MODEL,
    instructions=bot + " Use the python tool to analyse hdb_resale_prices.csv.",
    tools=[python_tool],
    input="What's the median 4-room resale price in Tampines, Bedok and Pasir Ris?",
)

print(response.output_text)

In [ ]:
# peek at the Python code the AI wrote
for item in response.output:
    if item.type == "code_interpreter_call":
        print(item.code)

> 💰 Each python tool session costs a few cents, and it stays open for 20 minutes after its last use.

## 4. Web search 🔍
For anything that changes — news, policies, opening hours.

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions=bot,
    tools=[{"type": "web_search"}],
    input="Any recent news about HDB resale prices?",
)

print(response.output_text)

## 5. Save the file ID for your bot 💾
Your bot (`main.py`) will use this same uploaded file, so we save its ID in `.env`.

In [ ]:
with open(".env", "a") as f:
    f.write(f"\nOPENAI_HDB_FILE_ID={hdb_file.id}\n")

print("Saved! Open .env to check.")

## ✏️ Exercises
1. Ask the python tool: *"Which town has the cheapest 5-room flats?"* and *"How did prices change month by month?"*
2. Give the AI **both** tools in one list: `tools=[python_tool, {"type": "web_search"}]`. Ask something that needs both.
3. 🌶️ Challenge: download another dataset from **data.gov.sg** (e.g. COE prices), upload it, and ask about it.

In [ ]:
# your code here